# Lab 08: Simple Linear Regression
> CLO3 | LLo: สร้าง SLR, ประเมิน SE/t-stat/p-value, วัด R²/RSE

## บทนำ

ใน Lab นี้เราจะฝึกสร้าง Simple Linear Regression (SLR) โดยใช้ข้อมูล Advertising dataset ที่ใช้ใน ISLP Chapter 3. เราจะเริ่มจากการคำนวณ β̂₀ และ β̂₁ **ด้วยมือ** โดยใช้สูตร least squares เพื่อให้เข้าใจที่มาของค่าเหล่านี้ จากนั้นจึงเปรียบเทียบผลกับ sklearn และ statsmodels. ผลทั้งสามวิธีต้องเท่ากัน — นั่นคือหลักฐานว่าเราเข้าใจ algorithm อย่างแท้จริง. ส่วนที่สำคัญที่สุดคือการอ่านและตีความ statsmodels summary table ซึ่งเป็นทักษะหลักของ Data Scientist ในการวิเคราะห์ regression.


In [ ]:
# ─── Import libraries ──────────────────────────────────────────────────────
# วัตถุประสงค์: โหลด library ที่จำเป็นสำหรับ regression analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

# ─── โหลดข้อมูล Advertising ────────────────────────────────────────────────
# วัตถุประสงค์: โหลด dataset จาก ISLP สำหรับ regression
# ถ้าไม่มีไฟล์ สร้าง synthetic data ที่มีคุณสมบัติใกล้เคียง ISLP
try:
    df = pd.read_csv('Advertising.csv')
    if 'Unnamed: 0' in df.columns:
        df = df.drop('Unnamed: 0', axis=1)
except FileNotFoundError:
    # สร้าง synthetic data แทน (β₀=7.03, β₁=0.0475, noise~N(0,3.26²))
    np.random.seed(42)
    n = 200
    TV = np.random.uniform(0.7, 296.4, n)
    Sales = 7.03 + 0.0475 * TV + np.random.normal(0, 3.26, n)
    df = pd.DataFrame({'TV': TV, 'Sales': Sales})
    print('ใช้ synthetic data (ไม่พบ Advertising.csv)')

print(f'ข้อมูล: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()


## Part 1: คำนวณ β̂₀ และ β̂₁ จากสูตร (From Scratch)

**Part นี้เราจะคำนวณสัมประสิทธิ์ด้วยมือ เพื่อให้เข้าใจที่มาของ Least Squares solution**

สูตร Least Squares:
- β̂₁ = Σ(xᵢ−x̄)(yᵢ−ȳ) / Σ(xᵢ−x̄)² = Cov(X,Y) / Var(X)
- β̂₀ = ȳ − β̂₁·x̄


In [ ]:
# ─── คำนวณ β̂₀ และ β̂₁ จากสูตร Least Squares ──────────────────────────────
# วัตถุประสงค์: ประมาณค่าสัมประสิทธิ์โดยใช้สูตร closed-form

x = df['TV'].values
y = df['Sales'].values

x_bar = np.mean(x)   # x̄
y_bar = np.mean(y)   # ȳ

# คำนวณ numerator และ denominator ของ β̂₁
numerator = np.sum((x - x_bar) * (y - y_bar))  # Σ(xᵢ-x̄)(yᵢ-ȳ)
denominator = np.sum((x - x_bar) ** 2)          # Σ(xᵢ-x̄)²

beta1_scratch = numerator / denominator
beta0_scratch = y_bar - beta1_scratch * x_bar

print('=== คำนวณ From Scratch ===')
print(f'x̄ = {x_bar:.4f}')
print(f'ȳ = {y_bar:.4f}')
print(f'Σ(xᵢ-x̄)(yᵢ-ȳ) = {numerator:.4f}')
print(f'Σ(xᵢ-x̄)² = {denominator:.4f}')
print(f'β̂₁ = {beta1_scratch:.6f}')
print(f'β̂₀ = {beta0_scratch:.6f}')


### 🎯 TODO 1: Fit SLR ด้วย sklearn (Easy)

เราต้องการเปรียบเทียบ β̂ from-scratch กับ sklearn เพราะต้องได้ค่าเดียวกัน

ให้คุณ Fit `LinearRegression` จาก sklearn โดยใช้ TV เป็น feature และ Sales เป็น target แล้ว print β̂₀ และ β̂₁

**คาดหวัง**: ค่าต้องเท่ากับ from-scratch ทุก decimal


In [ ]:
# TODO 1: Fit LinearRegression จาก sklearn
# Hint: ใช้ LinearRegression().fit(X, y) โดย X ต้องเป็น 2D array (.reshape(-1,1))
raise NotImplementedError('TODO 1')

# ตัวอย่าง output ที่คาดหวัง:
# sklearn β̂₁ = 0.04753... (ควรเท่ากับ from-scratch)
# sklearn β̂₀ = 7.0326... (ควรเท่ากับ from-scratch)


## Part 2: statsmodels OLS — Full Summary Table

**Part นี้เราจะ Fit SLR ด้วย statsmodels เพื่อดู full output รวม SE, t-statistic, p-value, CI, R², RSE**

statsmodels formula API ใช้ R-style formula: `'Sales ~ TV'` ทำให้อ่านง่ายและ interpret ได้ชัดเจน


In [ ]:
# ─── Fit SLR ด้วย statsmodels ──────────────────────────────────────────────
# วัตถุประสงค์: ดู full regression summary รวม SE, t, p-value, CI, R², RSE

model = smf.ols('Sales ~ TV', data=df).fit()
print(model.summary())


In [ ]:
# ─── ดึงค่า individual statistics ──────────────────────────────────────────
# วัตถุประสงค์: อธิบายความหมายของแต่ละ statistic จาก summary table

print('=== ค่าจาก statsmodels ===')
print(f"β̂₁ (TV coef) = {model.params['TV']:.6f}")
print(f"SE(β̂₁) = {model.bse['TV']:.6f}")
print(f"t-statistic = {model.tvalues['TV']:.4f}")
print(f"p-value = {model.pvalues['TV']:.6e}")
ci = model.conf_int().loc['TV']
print(f"95% CI = ({ci[0]:.6f}, {ci[1]:.6f})")
print(f"\nR² = {model.rsquared:.4f}")
rse = np.sqrt(model.ssr / model.df_resid)
print(f"RSE = {rse:.4f}")
print(f"\nตีความ:")
print(f"  TV เพิ่ม 1 พัน $ → Sales เพิ่ม {model.params['TV']*1000:.1f} หน่วย")
print(f"  R² = {model.rsquared:.3f} → TV อธิบาย {model.rsquared*100:.1f}% ของ variance ใน Sales")
print(f"  p-value ≈ 0 → Reject H₀ → β₁ ≠ 0")


### 🎯 TODO 2: ตีความ Confidence Interval (Medium)

เราต้องการให้นักศึกษาสามารถตีความ CI ได้อย่างถูกต้อง เพราะ CI มักถูกเข้าใจผิด

ให้คุณ: (1) คำนวณ 95% CI ด้วยมือโดยใช้สูตร β̂₁ ± 2·SE(β̂₁) และ (2) เขียน comment อธิบายความหมายของ CI ใน 2–3 ประโยค (เป็นภาษาไทยก็ได้)

**คาดหวัง**: CI จากมือควรใกล้เคียงกับ statsmodels


In [ ]:
# TODO 2: คำนวณ 95% CI ด้วยมือและตีความ
# Hint: ดึง model.params['TV'] และ model.bse['TV'] แล้วคำนวณ lower = beta1 - 2*SE, upper = beta1 + 2*SE
raise NotImplementedError('TODO 2')

# เขียน comment ตีความ CI ที่ได้:
# ตีความ: ...


## Part 3: Visualization — Scatter Plot + Fitted Line + Residual Plot

**Part นี้เราจะสร้าง visualization เพื่อตรวจสอบ model fit ด้วยตา และดู residuals**

Residual plot ช่วยตรวจสอบ assumptions ของ SLR — ถ้า residuals สุ่มกระจายรอบ 0 แสดงว่า assumption ผ่าน


In [ ]:
# ─── Scatter Plot + Fitted Regression Line ─────────────────────────────────
# วัตถุประสงค์: แสดงภาพ data points และ best-fit line

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Scatter + Fitted Line
x_line = np.linspace(df['TV'].min(), df['TV'].max(), 100)
y_line = model.params['Intercept'] + model.params['TV'] * x_line

ax1.scatter(df['TV'], df['Sales'], alpha=0.4, s=30, label='Data')
ax1.plot(x_line, y_line, 'r-', linewidth=2,
         label=f'ŷ = {model.params["Intercept"]:.2f} + {model.params["TV"]:.4f}·TV')
ax1.set_xlabel('TV Budget (พัน $)')
ax1.set_ylabel('Sales (พันหน่วย)')
ax1.set_title('SLR: TV → Sales')
ax1.legend()

# ─── Residual Plot ─────────────────────────────────────────────────────────
# วัตถุประสงค์: ตรวจสอบ assumption ว่า residuals กระจายสม่ำเสมอ

residuals = model.resid
fitted = model.fittedvalues

ax2.scatter(fitted, residuals, alpha=0.4, s=30)
ax2.axhline(y=0, color='r', linestyle='--', linewidth=1)
ax2.set_xlabel('Fitted Values (ŷ)')
ax2.set_ylabel('Residuals (eᵢ = yᵢ − ŷᵢ)')
ax2.set_title('Residual Plot')

plt.tight_layout()
plt.show()

print(f'Residual mean = {residuals.mean():.6f} (ควรใกล้ 0)')
print(f'Residual std = {residuals.std():.4f} (ควรใกล้ RSE)')


### 🎯 TODO 3: คำนวณ RSE และ R² จากสูตร (Medium)

เราต้องการให้นักศึกษาคำนวณ RSE และ R² ด้วยมือ เพราะการเข้าใจที่มาช่วยให้ตีความได้ถูกต้องและ debug ได้เมื่อเกิดปัญหา

ให้คุณ: คำนวณ RSS, TSS, RSE, R² จากสูตร แล้วเปรียบเทียบกับค่าจาก statsmodels

สูตร:
- RSS = Σ(yᵢ − ŷᵢ)²
- TSS = Σ(yᵢ − ȳ)²
- RSE = √(RSS/(n−2))
- R² = 1 − RSS/TSS

**คาดหวัง**: RSE ≈ 3.26, R² ≈ 0.612


In [ ]:
# TODO 3: คำนวณ RSS, TSS, RSE, R² จากสูตร
# Hint: fitted values = model.fittedvalues, y_true = df['Sales'].values
# RSS = np.sum((y_true - fitted)**2)
# TSS = np.sum((y_true - y_true.mean())**2)
raise NotImplementedError('TODO 3')


## สรุปผล Lab 08

| วิธี | β̂₀ | β̂₁ | R² | RSE |
|-----|-----|-----|-----|-----|
| From Scratch | ? | ? | — | — |
| sklearn | ? | ? | — | — |
| statsmodels | ? | ? | ? | ? |

**ข้อสังเกตสำคัญ**:
1. ทั้งสามวิธีให้ β̂ เหมือนกัน — Least Squares มี unique solution
2. statsmodels ให้ข้อมูล inference เพิ่มเติม (SE, t, p-value, CI) ซึ่ง sklearn ไม่มี
3. p-value ≈ 0 สำหรับ TV → มีหลักฐานแข็งแกร่งว่า TV มีผลต่อ Sales
4. R² = 0.612 → TV อธิบาย variance ได้ 61.2% — ยังเหลืออีก 38.8% ที่ต้องหา predictor เพิ่ม → Week 9
